# Prepare First-Pass SCEPTER Inputs

This notebook starts the SCEPTER part of the ERW MRV workflow. It assumes the cropland validation notebook has produced cleaned field parcels and uses those parcels as spatial model units.

At this stage we are not yet running SCEPTER. We are preparing clean, auditable input tables: one table for model units, one table for ERW scenarios, and one expanded run table with every model-unit/scenario combination.


## Workflow

1. **Load cleaned cropland parcels** from notebook `03_validate_cropland_outputs.ipynb`.
2. **Create model units** from those parcels, including area and centroid coordinates.
3. **Attach HWSD2 soil defaults and CHIRPS rainfall** prepared by notebooks `03b` and `03a`.
4. **Define ERW scenarios** such as baseline, 10 t/ha, 20 t/ha, and 40 t/ha basalt applications.
5. **Expand model runs** so every parcel/model unit is paired with every scenario.
6. **Write staging files** under `data/scepter_runs/inputs/`.

The soil and rainfall values are now read from processed project artifacts. Run notebooks `03a_prepare_monthly_rainfall.ipynb` and `03b_prepare_hwsd_soil_inputs.ipynb` before this notebook.


In [1]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely.geometry import box

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, ensure_dir
from erw_mrv.scepter import (
    DEFAULT_SCENARIOS,
    ScepterDefaults,
    expand_scepter_runs,
    load_cleaned_parcels,
    make_model_units,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/erw_spatial_mrv
SOURCE_PROJECT_ROOT = /content/drive/MyDrive/erw_spatial_mrv
DATA_ROOT = /content/drive/MyDrive/erw_spatial_mrv/data


## Configure Inputs

The preferred input is a cleaned parcel layer from notebook `03`. In the current cropland-only workflow, parcel layers are optional. If no parcel layer exists, this notebook builds coarse SCEPTER model units from blocks of the cropland raster written by notebook `02`.


In [2]:
OUTPUT_TAG = "jan_jun_2026"
LANDCOVER_DIR = DATA_ROOT / "processed" / "landcover"
SOIL_DEFAULTS_PATH = DATA_ROOT / "processed" / "soil" / "hwsd2" / "scepter_soil_defaults_hwsd2.csv"
RAINFALL_SUMMARY_PATH = DATA_ROOT / "processed" / "climate" / "rainfall" / "monthly_rainfall_aoi_jan_jun_2026.csv"

REQUIRE_REAL_SOIL = True
REQUIRE_REAL_RAINFALL = True
RUNOFF_FRACTION_OF_PRECIPITATION = 0.25
VALIDATION_DIR = LANDCOVER_DIR / "validation" / OUTPUT_TAG

CROPLAND_RASTER_PATH = LANDCOVER_DIR / f"cropland_pixels_{OUTPUT_TAG}_paper_adapted.tif"
CLEANED_PARCELS_PATH = VALIDATION_DIR / f"field_parcels_{OUTPUT_TAG}_cleaned.gpkg"
CANDIDATE_PARCELS_PATH = LANDCOVER_DIR / f"field_parcels_{OUTPUT_TAG}_paper_adapted.gpkg"

# Raster fallback settings. Each output unit is a block containing cropland pixels.
RASTER_BLOCK_SIZE = 512
MIN_CROPLAND_PIXELS_PER_UNIT = 25

if CLEANED_PARCELS_PATH.exists():
    INPUT_MODE = "cleaned_parcels"
    PARCELS_PATH = CLEANED_PARCELS_PATH
    print(f"Using cleaned parcels: {PARCELS_PATH}")
elif CANDIDATE_PARCELS_PATH.exists():
    INPUT_MODE = "candidate_parcels"
    PARCELS_PATH = CANDIDATE_PARCELS_PATH
    print(f"Using candidate parcels because cleaned parcels are not ready: {PARCELS_PATH}")
elif CROPLAND_RASTER_PATH.exists():
    INPUT_MODE = "cropland_raster_blocks"
    PARCELS_PATH = None
    print(f"Using cropland raster blocks: {CROPLAND_RASTER_PATH}")
else:
    raise FileNotFoundError(
        "No cropland input found. Run notebook 02 first. Checked: "
        f"{CLEANED_PARCELS_PATH}, {CANDIDATE_PARCELS_PATH}, and {CROPLAND_RASTER_PATH}"
    )

SCEPTER_INPUT_DIR = ensure_dir(SCEPTER_INPUTS)
print(f"SCEPTER input directory for notebook 04: {SCEPTER_INPUT_DIR}")
print("Notebook 04 writes only data/scepter_runs/inputs. Notebook 05 creates data/scepter_runs/runs and data/scepter_runs/outputs.")
SCEPTER_INPUT_DIR


Using cropland raster blocks: /content/drive/MyDrive/erw_spatial_mrv/data/processed/landcover/cropland_pixels_jan_jun_2026_paper_adapted.tif
SCEPTER input directory for notebook 04: /content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs
Notebook 04 writes only data/scepter_runs/inputs. Notebook 05 creates data/scepter_runs/runs and data/scepter_runs/outputs.


PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs')

## Load Spatial Model Units

This cell loads parcel polygons when available. Otherwise it creates coarse spatial units from cropland raster blocks. The raster-block fallback is intentionally lightweight for cloud execution and first-pass SCEPTER staging.


In [3]:
MAX_TEST_UNITS = 100  # set to None when ready to prepare every available unit


def load_cropland_raster_blocks(
    raster_path: Path,
    block_size: int = RASTER_BLOCK_SIZE,
    min_pixels: int = MIN_CROPLAND_PIXELS_PER_UNIT,
) -> gpd.GeoDataFrame:
    records = []
    with rasterio.open(raster_path) as src:
        pixel_area_m2 = abs(src.transform.a * src.transform.e)
        for row_start in range(0, src.height, block_size):
            height = min(block_size, src.height - row_start)
            for col_start in range(0, src.width, block_size):
                width = min(block_size, src.width - col_start)
                window = Window(col_start, row_start, width, height)
                data = src.read(1, window=window)
                cropland_pixels = int((data == 1).sum())
                if cropland_pixels < min_pixels:
                    continue

                left, bottom, right, top = src.window_bounds(window)
                records.append(
                    {
                        "value": 1,
                        "source": "cropland_raster_block",
                        "cropland_pixels": cropland_pixels,
                        "area_m2": cropland_pixels * pixel_area_m2,
                        "area_ha": (cropland_pixels * pixel_area_m2) / 10_000,
                        "geometry": box(left, bottom, right, top),
                    }
                )

        crs = src.crs

    if not records:
        raise ValueError(f"No cropland raster blocks found in {raster_path}")
    return gpd.GeoDataFrame(records, crs=crs)


if INPUT_MODE in {"cleaned_parcels", "candidate_parcels"}:
    parcels = load_cleaned_parcels(PARCELS_PATH)
else:
    parcels = load_cropland_raster_blocks(CROPLAND_RASTER_PATH)

print(f"Input mode: {INPUT_MODE}")
print(f"Loaded spatial units: {len(parcels):,}")
print(f"Total cropland area represented: {parcels['area_ha'].sum():,.2f} ha")
parcels[["area_ha"]].describe()


Input mode: cropland_raster_blocks
Loaded spatial units: 110
Total cropland area represented: 1,676,858.67 ha


,area_ha
count,110.000000
mean,15244.169727
std,8513.827908
min,31.140000
25%,8485.155000
50%,18891.270000
75%,23097.442500
max,23571.540000


## Create Model Units

These units include geometry, area, centroid coordinates, HWSD2 soil defaults, and CHIRPS rainfall. Runoff is still an explicit first-pass estimate derived from rainfall until a runoff layer is added.


In [4]:
def load_soil_defaults(path: Path, require_real: bool = REQUIRE_REAL_SOIL) -> tuple[dict, dict]:
    if not path.exists():
        message = f"HWSD2 soil defaults not found: {path}. Run notebook 03b first."
        if require_real:
            raise FileNotFoundError(message)
        print(f"Warning: {message} Using explicit placeholder soil values.")
        return {}, {"soil_source": "placeholder_defaults", "soil_source_path": str(path)}

    soil = pd.read_csv(path).iloc[0].to_dict()
    usable = {}
    for key in ["soil_ph", "cec_cmolc_kg", "clay_pct", "bulk_density_g_cm3", "soil_depth_cm"]:
        value = pd.to_numeric(soil.get(key), errors="coerce")
        if pd.notna(value):
            usable[key] = float(value)
    metadata = {
        "soil_source": str(soil.get("source", "hwsd2")),
        "soil_source_path": str(path),
        "soil_note": str(soil.get("note", "AOI-weighted HWSD2 defaults")),
    }
    print(f"Using HWSD2 soil defaults from: {path}")
    return usable, metadata


def load_rainfall_defaults(path: Path, require_real: bool = REQUIRE_REAL_RAINFALL) -> tuple[dict, dict]:
    if not path.exists():
        message = f"CHIRPS rainfall summary not found: {path}. Run notebook 03a first."
        if require_real:
            raise FileNotFoundError(message)
        print(f"Warning: {message} Using placeholder precipitation.")
        precipitation_mm_yr = 1200.0
        return (
            {"precipitation_mm_yr": precipitation_mm_yr, "runoff_mm_yr": precipitation_mm_yr * RUNOFF_FRACTION_OF_PRECIPITATION},
            {"rainfall_source": "placeholder_defaults", "rainfall_source_path": str(path), "rainfall_months_used": ""},
        )

    rainfall = pd.read_csv(path)
    if rainfall.empty or "rainfall_mm" not in rainfall.columns:
        raise ValueError(f"Rainfall summary is empty or missing rainfall_mm: {path}")

    rainfall_values = pd.to_numeric(rainfall["rainfall_mm"], errors="coerce").dropna()
    if rainfall_values.empty:
        raise ValueError(f"Rainfall summary has no numeric rainfall_mm values: {path}")

    months_used = rainfall["month"].astype(str).tolist() if "month" in rainfall.columns else []
    precipitation_mm_yr = float(rainfall_values.sum() * (12 / len(rainfall_values)))
    runoff_mm_yr = float(precipitation_mm_yr * RUNOFF_FRACTION_OF_PRECIPITATION)
    metadata = {
        "rainfall_source": "CHIRPS rainfall_chirps_monthly via DE Africa S3",
        "rainfall_source_path": str(path),
        "rainfall_months_used": ",".join(months_used),
        "missing_requested_months": str(rainfall.get("missing_requested_months", pd.Series([""])).iloc[0]),
        "runoff_note": f"runoff_mm_yr estimated as {RUNOFF_FRACTION_OF_PRECIPITATION:.2f} * precipitation_mm_yr",
    }
    print(f"Using CHIRPS rainfall summary from: {path}")
    print(f"Annualized precipitation: {precipitation_mm_yr:.1f} mm/yr")
    print(f"Estimated runoff: {runoff_mm_yr:.1f} mm/yr")
    return {"precipitation_mm_yr": precipitation_mm_yr, "runoff_mm_yr": runoff_mm_yr}, metadata


soil_defaults = {
    "soil_ph": 6.2,
    "cec_cmolc_kg": 14.0,
    "clay_pct": 28.0,
    "bulk_density_g_cm3": 1.30,
    "soil_depth_cm": 30.0,
}
loaded_soil_defaults, soil_metadata = load_soil_defaults(SOIL_DEFAULTS_PATH)
rainfall_defaults, rainfall_metadata = load_rainfall_defaults(RAINFALL_SUMMARY_PATH)
soil_defaults.update(loaded_soil_defaults)

defaults = ScepterDefaults(
    **soil_defaults,
    temperature_c=24.0,
    precipitation_mm_yr=rainfall_defaults["precipitation_mm_yr"],
    runoff_mm_yr=rainfall_defaults["runoff_mm_yr"],
    basalt_application_t_ha=20.0,
    basalt_d50_um=50.0,
    simulation_years=10,
)

model_units = make_model_units(parcels, defaults=defaults, max_units=MAX_TEST_UNITS)
model_units["input_status"] = "hwsd2_soil_chirps_rainfall_runoff_estimate"

if INPUT_MODE == "cropland_raster_blocks":
    selected_area = parcels.sort_values("area_ha", ascending=False).reset_index(drop=True)
    if MAX_TEST_UNITS is not None:
        selected_area = selected_area.head(MAX_TEST_UNITS).copy()
    model_units["block_geometry_area_m2"] = model_units.geometry.area
    model_units["area_m2"] = selected_area["area_m2"].to_numpy()
    model_units["area_ha"] = selected_area["area_ha"].to_numpy()
    model_units["cropland_pixels"] = selected_area["cropland_pixels"].to_numpy()
    model_units["input_status"] = "cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate"

for key, value in {**soil_metadata, **rainfall_metadata}.items():
    model_units[key] = value

MODEL_UNIT_COLUMNS = [
    "model_unit_id",
    "area_ha",
    "centroid_lon",
    "centroid_lat",
    "soil_ph",
    "cec_cmolc_kg",
    "clay_pct",
    "bulk_density_g_cm3",
    "soil_depth_cm",
    "temperature_c",
    "precipitation_mm_yr",
    "runoff_mm_yr",
    "input_status",
    "soil_source",
    "soil_source_path",
    "soil_note",
    "rainfall_source",
    "rainfall_source_path",
    "rainfall_months_used",
    "missing_requested_months",
    "runoff_note",
]


def build_model_units_table(units: gpd.GeoDataFrame) -> pd.DataFrame:
    missing = [column for column in MODEL_UNIT_COLUMNS if column not in units.columns]
    if missing:
        raise ValueError(f"Model units are missing required real-data columns: {missing}")
    return pd.DataFrame(units[MODEL_UNIT_COLUMNS]).copy()


model_unit_table = build_model_units_table(model_units)

print(f"Model units prepared: {len(model_units):,}")
model_unit_table.head()


Using HWSD2 soil defaults from: /content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv
Using CHIRPS rainfall summary from: /content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv
Annualized precipitation: 1216.3 mm/yr
Estimated runoff: 304.1 mm/yr
Model units prepared: 100


,model_unit_id,area_ha,centroid_lon,centroid_lat,soil_ph,cec_cmolc_kg,clay_pct,bulk_density_g_cm3,soil_depth_cm,temperature_c,precipitation_mm_yr,runoff_mm_yr,input_status,soil_source,soil_source_path,soil_note,rainfall_source,rainfall_source_path,rainfall_months_used,missing_requested_months,runoff_note
0,mu_00001,23571.54,30.603259,0.703783,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
1,mu_00002,23554.44,30.603181,0.842627,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
2,mu_00003,23545.71,30.741118,0.842710,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
3,mu_00004,23543.10,30.603323,0.564939,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr
4,mu_00005,23541.12,30.741191,0.703852,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr


## Define Scenarios

The scenarios table controls amendment rate, particle size, and simulation length. The baseline scenario is important because ERW results should be compared against no-amendment conditions.


In [5]:
scenarios = DEFAULT_SCENARIOS.copy()
scenarios


,scenario_id,basalt_application_t_ha,basalt_d50_um,simulation_years,description
0,baseline_no_erw,0.0,50.0,10,No rock amendment baseline.
1,erw_10t_fine,10.0,50.0,10,Low application rate with fine basalt.
2,erw_20t_fine,20.0,50.0,10,Middle application rate with fine basalt.
3,erw_40t_medium,40.0,100.0,10,Higher application rate with medium basalt.


## Expand To Run Table

This creates one row for every model unit and scenario. Later, each `run_id` can map to a SCEPTER configuration folder or input file.


In [6]:
runs = expand_scepter_runs(model_unit_table, scenarios)
print(f"SCEPTER runs staged: {len(runs):,}")
runs.head()


SCEPTER runs staged: 400


,run_id,model_unit_id,scenario_id,area_ha,centroid_lon,centroid_lat,soil_ph,cec_cmolc_kg,clay_pct,bulk_density_g_cm3,soil_depth_cm,temperature_c,precipitation_mm_yr,runoff_mm_yr,input_status,soil_source,soil_source_path,soil_note,rainfall_source,rainfall_source_path,rainfall_months_used,missing_requested_months,runoff_note,basalt_application_t_ha,basalt_d50_um,simulation_years,description
0,mu_00001__baseline_no_erw,mu_00001,baseline_no_erw,23571.54,30.603259,0.703783,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr,0.0,50.0,10,No rock amendment baseline.
1,mu_00001__erw_10t_fine,mu_00001,erw_10t_fine,23571.54,30.603259,0.703783,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr,10.0,50.0,10,Low application rate with fine basalt.
2,mu_00001__erw_20t_fine,mu_00001,erw_20t_fine,23571.54,30.603259,0.703783,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr,20.0,50.0,10,Middle application rate with fine basalt.
3,mu_00001__erw_40t_medium,mu_00001,erw_40t_medium,23571.54,30.603259,0.703783,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-02,2026-03,2026-04,2026-05",2026-06,runoff_mm_yr estimated as 0.25 * precipitation_mm_yr,40.0,100.0,10,Higher application rate with medium basalt.
4,mu_00002__baseline_no_erw,mu_00002,baseline_no_erw,23554.44,30.603181,0.842627,6.2,14.0,27.87396,1.3,30.0,24.0,1216.298121,304.07453,cropland_raster_block_with_hwsd2_soil_chirps_rainfall_runoff_estimate,/content/drive/MyDrive/erw_spatial_mrv/data/raw/soilgrids/HWSD2.mdb,/content/drive/MyDrive/erw_spatial_mrv/data/processed/soil/hwsd2/scepter_soil_defaults_hwsd2.csv,AOI pixel-share weighted HWSD2 defaults for first-pass SCEPTER staging,CHIRPS rainfall_chirps_monthly via DE Africa S3,/content/drive/MyDrive/erw_spatial_mrv/data/processed/climate/rainfall/monthly_rainfall_aoi_jan_jun_2026.csv,"2026-01,2026-0

## Write SCEPTER Staging Inputs


In [7]:
def write_real_scepter_inputs(
    units: gpd.GeoDataFrame,
    units_table: pd.DataFrame,
    runs_table: pd.DataFrame,
    scenarios_table: pd.DataFrame,
    output_dir: Path,
) -> dict[str, Path]:
    output_dir = ensure_dir(output_dir)

    units_gpkg_path = output_dir / "scepter_model_units.gpkg"
    units_table_path = output_dir / "scepter_model_units.csv"
    scenarios_path = output_dir / "scepter_scenarios.csv"
    runs_path = output_dir / "scepter_runs.csv"
    readme_path = output_dir / "README_scepter_inputs.md"

    units.to_file(units_gpkg_path, layer="model_units", driver="GPKG")
    units_table.to_csv(units_table_path, index=False)
    scenarios_table.to_csv(scenarios_path, index=False)
    runs_table.to_csv(runs_path, index=False)
    readme_path.write_text(
        "# SCEPTER Input Tables\n\n"
        "These files are generated by notebook 04 using real HWSD2 soil defaults and CHIRPS rainfall.\n\n"
        "- `scepter_model_units.gpkg`: spatial model units.\n"
        "- `scepter_model_units.csv`: non-spatial model-unit attributes with input provenance.\n"
        "- `scepter_scenarios.csv`: ERW application scenarios.\n"
        "- `scepter_runs.csv`: one row per model unit and scenario, carrying HWSD2/CHIRPS metadata.\n",
        encoding="utf-8",
    )
    return {
        "model_units_gpkg": units_gpkg_path,
        "model_units_csv": units_table_path,
        "scenarios_csv": scenarios_path,
        "runs_csv": runs_path,
        "readme": readme_path,
    }


written = write_real_scepter_inputs(
    model_units,
    model_unit_table,
    runs,
    scenarios,
    SCEPTER_INPUT_DIR,
)

required_metadata_columns = ["soil_source", "rainfall_source", "rainfall_months_used", "runoff_note"]
missing_files = [label for label, path in written.items() if not Path(path).exists()]
if missing_files:
    raise FileNotFoundError(f"Notebook 04 did not write expected SCEPTER input files: {missing_files}")

written_runs = pd.read_csv(written["runs_csv"])
written_units = pd.read_csv(written["model_units_csv"])
missing_run_columns = [column for column in required_metadata_columns if column not in written_runs.columns]
missing_unit_columns = [column for column in required_metadata_columns if column not in written_units.columns]
if missing_run_columns or missing_unit_columns:
    raise ValueError(
        "Notebook 04 wrote SCEPTER files, but real input metadata columns are missing. "
        f"Missing from runs: {missing_run_columns}; missing from units: {missing_unit_columns}"
    )

print("Wrote SCEPTER input files:")
for label, path in written.items():
    print(f"- {label}: {path}")
print(f"Staged model units: {len(written_units):,}")
print(f"Staged SCEPTER runs: {len(written_runs):,}")
print("Real input metadata columns verified in scepter_runs.csv.")
written


Wrote SCEPTER input files:
- model_units_gpkg: /content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.gpkg
- model_units_csv: /content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.csv
- scenarios_csv: /content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_scenarios.csv
- runs_csv: /content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_runs.csv
- readme: /content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/README_scepter_inputs.md
Staged model units: 100
Staged SCEPTER runs: 400
Real input metadata columns verified in scepter_runs.csv.


{'model_units_gpkg': PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.gpkg'),
 'model_units_csv': PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_model_units.csv'),
 'scenarios_csv': PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_scenarios.csv'),
 'runs_csv': PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/scepter_runs.csv'),
 'readme': PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/scepter_runs/inputs/README_scepter_inputs.md')}

## Quick QA

Check that every scenario has the same number of model units and that no required values are missing.


In [8]:
qa_by_scenario = runs.groupby("scenario_id").agg(
    run_count=("run_id", "count"),
    total_area_ha=("area_ha", "sum"),
    mean_application_t_ha=("basalt_application_t_ha", "mean"),
).reset_index()

required_columns = [
    "run_id",
    "model_unit_id",
    "scenario_id",
    "area_ha",
    "soil_ph",
    "temperature_c",
    "precipitation_mm_yr",
    "basalt_application_t_ha",
    "basalt_d50_um",
    "simulation_years",
]
missing_values = runs[required_columns].isna().sum().reset_index()
missing_values.columns = ["column", "missing_count"]

qa_by_scenario, missing_values


(       scenario_id  run_count  total_area_ha  mean_application_t_ha
 0  baseline_no_erw        100     1672522.38                    0.0
 1     erw_10t_fine        100     1672522.38                   10.0
 2     erw_20t_fine        100     1672522.38                   20.0
 3   erw_40t_medium        100     1672522.38                   40.0,
                     column  missing_count
 0                   run_id              0
 1            model_unit_id              0
 2              scenario_id              0
 3                  area_ha              0
 4                  soil_ph              0
 5            temperature_c              0
 6      precipitation_mm_yr              0
 7  basalt_application_t_ha              0
 8            basalt_d50_um              0
 9         simulation_years              0)

## Outputs From This Notebook

This notebook writes first-pass SCEPTER input staging files under `data/scepter_runs/inputs/`:

- `scepter_model_units.gpkg`: spatial cropland model units derived from cleaned parcel polygons when available, or coarse cropland raster blocks otherwise.
- `scepter_model_units.csv`: non-spatial model unit table with area, centroid, soil, and climate attributes.
- `scepter_scenarios.csv`: baseline and ERW application scenarios.
- `scepter_runs.csv`: one row per model unit and scenario, with a unique `run_id`.
- `README_scepter_inputs.md`: notes explaining the staging files and current assumptions.

Raster-block units are a cloud-friendly first-pass fallback. Replace them with cleaned field parcels later if you need parcel-level MRV precision.
